In [19]:
import pandas as pd
import numpy as np
import duckdb
import pickle

In [2]:
import sys
import os
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))
sys.path.insert(0, project_root)

from src.custom_metrics import precision_at_k, recall_at_k, ndcg_at_k

In [3]:
con = duckdb.connect()
df = con.execute(f"select * from '{"../data/interactions_final.parquet"}'").df()

In [4]:
val = df[df["split"] == "val"].copy()
train = df[df["split"] == "train"].copy()

# Ground truth 

relevant books for every user in validation part

In [5]:
gt = val[val['rating'] >= 4].groupby('user_id')['parent_asin'].apply(set).to_dict()

In [6]:
print(f"users in val with rating >= 4: {len(gt)}")
print(f"average number of books per such user: {np.mean([len(v) for v in gt.values()]):.2f}")

users in val with rating >= 4: 91497
average number of books per such user: 1.78


# Random baseline

making random predictions for all users simultaneously using vectorization

## Vectorization

In [7]:
all_items = train["parent_asin"].unique()
n_items = len(all_items)
item_to_idx = {item: i for i, item in enumerate(all_items)}

In [8]:
user_ids = list(gt.keys())
n_users = len(user_ids)

In [9]:
np.random.seed(42)
K = 10

In [10]:
gt_arrays = [np.array([item_to_idx[x] for x in gt[u]]) for u in user_ids]  #array of indeces of relevant books for a user
max_gt = max(len(a) for a in gt_arrays)

gt_padded = np.full((n_users, max_gt), -1)  # -1 padding
gt_sizes = np.zeros(n_users)   # true sizes
for u, arr in enumerate(gt_arrays):  # filling in values
    gt_padded[u, :len(arr)] = arr
    gt_sizes[u] = len(arr)
print(f"n_users = {n_users}, maximum relevant books per user = {max_gt}, average = {gt_sizes.mean():.2f}")

n_users = 91497, maximum relevant books per user = 98, average = 1.78


In [11]:
# 10 random books for every user
rand_rec_idx = np.random.randint(0, n_items, size=(n_users, K))

In [12]:
match_matrix = (rand_rec_idx[:, :, None] == gt_padded[:, None, :]) #(n_users, K, max_gt) for every user comparing each recom (10) with each relevant book (98)
match_per_gt = match_matrix.any(axis=2)   #(n_users, K) checking if at least 1 recom matched gt for a user
matches = match_per_gt.sum(axis=1).astype(np.float64)  # (n_users,) number of matches per user

In [13]:
print(f'overall matches between random and ground truth: {matches.sum():.0f}')

overall matches between random and ground truth: 7


## Metrics

In [14]:
discounts = 1.0 / np.log2(np.arange(1, K + 1) + 1) # (K,)
dcg = (match_per_gt * discounts).sum(axis=1)  # (n_users,)
idcg = np.zeros(n_users)
for k in range(1, K + 1):
    idcg += (gt_sizes >= k) * discounts[k - 1]  # only users with >= k relevant books

In [15]:
print("Random baseline:")
print(f"Precision@10: {(matches / K).mean():.8f}")
print(f"Recall@10: {(matches / gt_sizes).mean():.8f}")
print(f"NDCG@10: {(dcg / idcg).mean():.8f}")

Random baseline:
Precision@10: 0.00000765
Recall@10: 0.00004226
NDCG@10: 0.00001861


the metrics are expectedly low because out of 308k books we have only 1.78 relevant on average. and the chance of guessing it correctly is extremely small. such low metrics on random guessing also indicate that there is no data leakage from train to val

# Popularity baseline

recommending N most popular books to every user

In [16]:
popularity = (train.groupby("parent_asin")["user_id"].nunique().sort_values(ascending=False))
top_100 = popularity.head(100).index.tolist()

In [17]:
pop_scores = []
for user_id, rel in gt.items():  
    pop_scores.append({
        "Precision@10": precision_at_k(top_100, rel),
        "Recall@10": recall_at_k(top_100, rel),
        "NDCG@10": ndcg_at_k(top_100, rel),
    })
pop_df = pd.DataFrame(pop_scores).dropna()
print("Popularity baseline:")
pop_df.mean()   # avg among all users scores

Popularity baseline:


Precision@10    0.000405
Recall@10       0.002704
NDCG@10         0.001664
dtype: float64

# Baseline Results 

| Model | Precision@10 | Recall@10 | NDCG@10 |
|---|---|---|---|
| Random | 0.000007 | 0.000042 | 0.000018 |
| Popularity | 0.000405 | 0.002704 | 0.001664 |
| Difference | x81 | x64 | x92 |

- popularity gives improvement over random - confirms a real signal exists
- absolute values are low due to extreme long-tail distribution:
  - top-10 books cover only 0.9% of interactions
  - median book has 5 unique users
  - average user has only 1.78 relevant books in val
- personalized models must be significantly better than non-personalized popularity baseline

In [18]:
con.close()

# Saving artifacts  

In [21]:
idx_to_item = {i: item for item, i in item_to_idx.items()}
user_to_idx = {u: i for i, u in enumerate(user_ids)}
idx_to_user = {i: u for u, i in user_to_idx.items()}

In [ ]:
eval_setup = {
    "gt_padded": gt_padded,
    "gt_sizes": gt_sizes,
    "user_ids": user_ids,
    "user_to_idx": user_to_idx,
    "item_to_idx": item_to_idx,
    "idx_to_item": idx_to_item,
    "all_items": all_items,
    "n_users": n_users,
    "n_items": n_items,
    "K": K,
    "discounts": 1.0 / np.log2(np.arange(1, K + 1) + 1), 
    "max_gt": max_gt,
}

In [41]:
with open("../data/artifacts/eval_setup.pkl", "wb") as f:
    pickle.dump(eval_setup, f, protocol=pickle.HIGHEST_PROTOCOL)